In [32]:
import numpy as np
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import classification_report
import random

In [2]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [3]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [5]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [6]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [7]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

In [8]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [9]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'label': sentiment}

In [10]:
ds_russian = ds_russian.map(calc_sentinent)

In [11]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 111667
})

In [12]:
num_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).num_rows
num_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).num_rows
num_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).num_rows

In [13]:
print('Number of negative reviews: {}'.format(num_negative))
print('Number of neutral reviews: {}'.format(num_neutral))
print('Number of positive reviews: {}'.format(num_positive))

Number of negative reviews: 3788
Number of neutral reviews: 3227
Number of positive reviews: 104652


In [14]:
df_russian_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).shuffle(seed=42).select(range(num_negative))
df_russian_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).shuffle(seed=42).select(range(num_neutral))
df_russian_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).shuffle(seed=42).select(range(num_negative)) #YES, we reduce numbers

In [15]:
df_russian_reduced = concatenate_datasets([df_russian_negative, df_russian_neutral, df_russian_positive]).shuffle(seed=42)

In [16]:
df_russian_reduced

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 10803
})

In [17]:
ds_russian_splitted = df_russian_reduced.train_test_split(test_size=0.1, seed=42)

In [18]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

In [33]:
def predict_sentiment(row):
    return {
        'predicted': random.randint(0, 2)
    }

In [34]:
ds_russian_splitted_with_prediction = ds_russian_splitted['test'].map(predict_sentiment)

Map:   0%|          | 0/1081 [00:00<?, ? examples/s]

In [35]:
ds_russian_splitted_with_prediction

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label', 'predicted'],
    num_rows: 1081
})

In [36]:
print(classification_report(ds_russian_splitted_with_prediction["label"], ds_russian_splitted_with_prediction["predicted"], target_names=[id2label[i] for i in range(NUM_LABELS)]))

              precision    recall  f1-score   support

    negative       0.36      0.33      0.34       395
     neutral       0.25      0.28      0.26       319
    positive       0.32      0.32      0.32       367

    accuracy                           0.31      1081
   macro avg       0.31      0.31      0.31      1081
weighted avg       0.31      0.31      0.31      1081

